In [28]:
import torch
# from tqdm.notebook import tqdm
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [29]:
# Used to create token ids, encode data, and decode tokens
class Processor:
    def __init__(self):
        self.encodings = {}
        self.decodings = {}

    # Read data to create token ids
    def ingest(self, data=str):
        raw_chars = list(data)
        unique_chars = sorted(list(set(raw_chars)))

        # Assign each char to a token id
        for token_id, u_char in enumerate(unique_chars):
            self.encodings[u_char]    = token_id
            self.decodings[token_id] = u_char

        # Assign vocab size
        self.vocab_size = len(unique_chars)

    # Encode characters to token ids
    def encode(self, data=str):
        raw_chars = list(data)
        tokens = [self.encodings[raw_char] for raw_char in raw_chars]
        return tokens
    
    # Decode tokens into characters
    def decode(self, data=list):
        decoded_tokens = [self.decodings[token_id] for token_id in data]
        decoded_string = "".join(decoded_tokens)
        return decoded_string

In [ ]:
# Creates instance of one model
# All hyperparameters and training are done inside this object
class Model():
    def __init__(self, vocab_size, B=256, T=128, C=64, H=64, lr=1e-3, beta1=0.9, beta2=0.999, epsilon=1e-8):
        # Define hyper parameters
        self.B = B   # Batch size
        self.T = T   # Sequence length or Block size
        self.C = C   # Embedding dimension
        self.H = H   # Matches C as we only are implementing one head
        self.lr = lr # Learning rate
        self.vocab_size = vocab_size

        # Adam optimizer hyperparameters
        self.beta1 = beta1      # Used for 1st moment weights
        self.beta2 = beta2      # Used for 2nd moment weights
        self.epsilon = epsilon  # Used for numerical stability in weight update
        self.t = 0              # Timestep for bias correction

        # Define matrices
        self.embedding_matrix = torch.randn(vocab_size, C, device = device) / (self.C ** 0.5)  # Holds embedding vectors for each token (Vocab_Size x C)
        self.W_q = torch.randn(H, C, device = device) / (self.C ** 0.5)    # Holds the Query weights (What we look for given input)
        self.W_k = torch.randn(H, C, device = device) / (self.C ** 0.5)    # Holds the Key weights (What the input holds/represents or has to offer)
        self.W_v = torch.randn(H, C, device = device) / (self.C ** 0.5)    # Holds the Value weights (Content that should be passed forward)
        ### Since our Head size is the same as our Embedding dimension, we can use the embedding matrix as our lm_head matrix
        ### I chose (H x C) dimensions because torches nn.Linear stores the parameter dimensions backwards like above
        ### This allows for similar computation with transposing the weights

        # Positional embedding to allow attention to grasp positional context of words rather than treating all the words in a context window the same (positionally)
        self.W_pos = torch.randn(T, C, device=device) / (self.C ** 0.5)

        # Weight matrices for our feed-forward network layers + their bias vectors
        # First layer
        self.W_ff1 = torch.randn(4*C, C, device=device) / (self.C ** 0.5)
        self.b_ff1 = torch.randn(4*C, device=device) / (self.C ** 0.5)

        # Second layer
        self.W_ff2 = torch.randn(C, 4*C, device=device) / (self.C ** 0.5)
        self.b_ff2 = torch.randn(C, device=device) / (self.C ** 0.5)
        

        # Adam optimizer moment matrices (4 weight matrices -> 4 m and 4 v total)
        self.m_Wq = torch.zeros_like(self.W_q)
        self.v_Wq = torch.zeros_like(self.W_q)

        self.m_Wk = torch.zeros_like(self.W_k)
        self.v_Wk = torch.zeros_like(self.W_k)

        self.m_Wv = torch.zeros_like(self.W_v)
        self.v_Wv = torch.zeros_like(self.W_v)

        self.m_Emb = torch.zeros_like(self.embedding_matrix)
        self.v_Emb = torch.zeros_like(self.embedding_matrix)

        self.m_Wpos = torch.zeros_like(self.W_pos)
        self.v_Wpos = torch.zeros_like(self.W_pos)

        ### Feed forward matrices monments
        self.m_Wff1 = torch.zeros_like(self.W_ff1)
        self.v_Wff1 = torch.zeros_like(self.W_ff1)
        self.m_bff1 = torch.zeros_like(self.b_ff1)
        self.v_bff1 = torch.zeros_like(self.b_ff1)

        self.m_Wff2 = torch.zeros_like(self.W_ff2)
        self.v_Wff2 = torch.zeros_like(self.W_ff2)
        self.m_bff2 = torch.zeros_like(self.b_ff2)
        self.v_bff2 = torch.zeros_like(self.b_ff2)


    # Saving weights
    def save_weights(self, filepath="model_weights.pt"):
        weights = {
            "W_q": self.W_q.cpu(),
            "W_k": self.W_k.cpu(),
            "W_v": self.W_v.cpu(),
            "embedding_matrix": self.embedding_matrix.cpu(),
            "W_pos": self.W_pos.cpu(),
            "W_ff1": self.W_ff1.cpu(),
            "b_ff1": self.b_ff1.cpu(),
            "W_ff2": self.W_ff2.cpu(),
            "b_ff2": self.b_ff2.cpu(),
            "hyperparams": {
                "vocab_size": self.vocab_size,
                "B": self.B,
                "T": self.T,
                "C": self.C,
                "H": self.H,
                "lr": self.lr,
            }
        }
        torch.save(weights, filepath)
        print(f"Weights successfully saved to {filepath}")

    # Loading saved weights
    def load_weights(self, filepath="model_weights.pt"):
        checkpoint = torch.load(filepath, map_location=device)
        
        self.W_q = checkpoint["W_q"].to(device)
        self.W_k = checkpoint["W_k"].to(device)
        self.W_v = checkpoint["W_v"].to(device)
        self.embedding_matrix = checkpoint["embedding_matrix"].to(device)
        self.W_pos = checkpoint["W_pos"].to(device)
        self.W_ff1 = checkpoint["W_ff1"].to(device)
        self.b_ff1 = checkpoint["b_ff1"].to(device)
        self.W_ff2 = checkpoint["W_ff2"].to(device)
        self.b_ff2 = checkpoint["b_ff2"].to(device)

        # Restore saved hyperparams so shapes match
        hp = checkpoint["hyperparams"]
        self.B, self.T, self.C, self.H = hp["B"], hp["T"], hp["C"], hp["H"]

        # Re-Initialize Adam optimizer matrices
        self.t = 0
        self.m_Wq = torch.zeros_like(self.W_q)
        self.v_Wq = torch.zeros_like(self.W_q)
        self.m_Wk = torch.zeros_like(self.W_k)
        self.v_Wk = torch.zeros_like(self.W_k)
        self.m_Wv = torch.zeros_like(self.W_v)
        self.v_Wv = torch.zeros_like(self.W_v)
        self.m_Emb = torch.zeros_like(self.embedding_matrix)
        self.v_Emb = torch.zeros_like(self.embedding_matrix)
        self.m_Wpos = torch.zeros_like(self.W_pos)
        self.v_Wpos = torch.zeros_like(self.W_pos)
        
        print(f"Weights successfully loaded from {filepath}")

    # Gets next token from given text
    def generate_from_text(self, text, encode_fn):
        # Convert text to token ids
        tokens = encode_fn(text)

        # Crop length if too long
        tokens = tokens[-self.T:]

        # Get B = 1
        x_ids = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0) # (1, T)
        # Get embedding vectors
        X = self.embedding_matrix[x_ids]

        # Holding copy of T temporarily
        original_T = self.T
        self.T = x_ids.shape[1]

        # Run forward pass for generation
        next_token_id = self.forward_pass(X, generation=True)

        # Restore T
        self.T = original_T

        return next_token_id.item()

    # Generate max new tokens
    def generate(self, prompt_text, processor, max_new_tokens=30):
        curr_text = prompt_text
        
        for _ in range(max_new_tokens):
            next_id = self.generate_from_text(curr_text, processor.encode)
            next_char = processor.decode([next_id])
            curr_text += next_char
            
        return curr_text

    # Create (B, T, C) matrix of randomly selected tokens from given data
    def build_train_batch(self, data):
        indices = torch.randint(len(data) - self.T - 1, (self.B,), device = device) # (B, 1)
        x_ids = torch.stack([data[ix:ix+self.T] for ix in indices]) # (B, T)
        y = torch.stack([data[ix+1:ix+self.T+1] for ix in indices]) # (B, T)

        # Replace token ids with their embedding vectors
        X = self.embedding_matrix[x_ids]    # (B, T, C)

        return x_ids, X, y
        
    # Calculate Q and K to get our pre-softmax attention matrix (A)
    def get_affinities(self, X):
        # Get our Query and Key matrices
        self.Q = X @ self.W_q.T    # (B, T, C) @ (C, H) --> (B, T, H)
        self.K = X @ self.W_k.T    # (B, T, C) @ (C, H) --> (B, T, H)
        ### This moves from the embedding dimension to our head size dimension
        ### In our case the head size is equal to the embedding dimension, so not much change happens here

        self.A = self.Q @ self.K.transpose(-2, -1)     # (B, T, H) @ (B, H, T) --> (B, T, T)
        self.A = self.A / (self.H ** 0.5)       # Scaling to prevent crazy value growth
        ### I use .tranpose here to manually swap dimension -2 and -1, or T and H, to allow correct matrix multiplication
        

    # Given we have our affinities, A, we now turn it to a lower triangle and softmax
    # We do so by setting the upper triangle to -inf
    # This ensures the softmax excludes future tokens, preventing a token from looking into the 'future'
    def softmax_attention(self):
        seq_len = self.A.shape[-1]

        # Generate a lower triangle of ones - Then set 0's to -infinity
        tril = torch.tril(torch.ones(seq_len, seq_len, device = device))
        A_shifted = self.A - self.A.max(dim=-1, keepdim=True).values  # Shifts A by subtracting max of each row to each element (Prevents overflow cases)
        A_masked = A_shifted.masked_fill(tril == 0, float('-inf'))  # --> (B, T, T) with only lower triangles maintained

        # Exponentiate all elements
        exp_vals = torch.exp(A_masked)
        exp_row_sums = exp_vals.sum(dim=-1, keepdim=True)    # Get the sums of each row (dim=-1 or dim=2)

        # Calculate softmaxes by dividing each exponentiated value by its rows sum of exponentiates 
        self.S = exp_vals / exp_row_sums   
        ### S is still --> (B, T, T) 

    # This is where our 'learning' is retrieved. 
    # Using the softmaxed attention values, S, we take a weighted average of the 'content'
    def value_aggregation(self, X):
        # Get our Value matrix
        self.V = X @ self.W_v.T      # (B, T, C) @ (C, H) --> (B, T, H)

        # Obtain our output before undoing projection
        self.O = self.S @ self.V   # (B, T, T) @ (B, T, H) --> (B, T, H)

        ### This is now commented out as O must go through the 2 layer feed forward MLP before creating our logits
        # Bring our output back to C dim and get logits
        #self.Z = self.O @ self.embedding_matrix.T     # (B, T, H) @ (C, Vocab size) --> (B, T, Vocab Size)
        ### The only reason I used embedding matrix here is because H == C
        ### When the head size does NOT equal C, I must add a lm_head matrix of dimenion (Vocab size, H)

    # Runs our Output through a 2 layer MLP - Feed-Forward Layer
    def FFL(self):
        # First layer --> O_ff1
        self.O_ff1 = self.O @ self.W_ff1.T + self.b_ff1     # (B, T, C) @ (C, 4C) + (4C,) = (B, T, 4C)

        # ReLU Activation --> O_ff2
        self.O_ff2 = torch.relu(self.O_ff1)     # (B, T, 4C)

        # Second layer
        self.O_hat = self.O_ff2 @ self.W_ff2.T + self.b_ff2 # (B, T, 4C) @ (4C, C) + (C,) = (B, T, C)

        # Pass this to create our logits
        self.Z = self.O_hat @ self.embedding_matrix.T

    # Softmax for our probabilities of next token
    def logits_to_p(self):
        # Exponentiate all elements
        Z_shifted = self.Z - self.Z.max(dim=-1, keepdim=True).values # Shift for overflow case
        exp_vals = torch.exp(Z_shifted)
        exp_row_sums = exp_vals.sum(dim=-1, keepdim=True)    # Get the sums of each row (dim=-1 or dim=2)

        # Calculate softmaxes by dividing each exponentiated value by its rows sum of exponentiates 
        self.P = exp_vals / exp_row_sums     # (B, T, Vocab_size)

        # Safe gaurd to prevent nan values from being used
        self.P = torch.nan_to_num(self.P, nan=1.0 / self.vocab_size)
        ### Now we have our probabilities for the next token of each token in each batch

    # Perform one forward pass to calculate predicted output
    def forward_pass(self, X, generation=False):      # X is expected (B, T, C)
        # Add positional embedding matrix to X (For positional awareness)
        token_count = X.shape[1]
        X = X + self.W_pos[:token_count]

        self.get_affinities(X)      # Q @ K.T  
        self.softmax_attention()    # Softmax(A)
        self.value_aggregation(X)   # O = S @ V 
        self.FFL()                  # O --> 2 Layer MLP --> Z = O_hat @ W_emb.T
        self.logits_to_p()          # Softmax(Z)

        # If we would like the next token to be returned
        if generation:
            last_p = self.P[:, -1, :]   # (B, vocab size)
            next_token_ids = torch.multinomial(last_p, num_samples=1)
            return next_token_ids


    
    ##### WEIGHT UPDATING #####
    # Must be called first - GZ is defined here and used in other gradients
    def gradient_W_v(self, X):
        # X - (B, T, C)
        # A - (B, T, T)
        # P - (B, T, Vocab size)
        # Y - (B, T)
        # W_E or W_lm - (Vocab size, C)
        # Gradient, or G, of A = Derivative of Loss wrt. A
        # Our softmaxes are based off our logits, Z = O @ W_lm
        # We know:
        # GZ = P - Y
        # GO = GZ @ W_E     (B, T, Vocab size) @ (Vocab size, H) --> (B, T, H)
        # GV = S^T @ GO     (B, T, T) @ (B, T, H) --> (B, T, H)
        # GW_V = GV^T @ X   (B, H, T) @ (B, T, C) --> (H, C) (Averaged over B)
        # GW_V = (GO^T @ S) @ X
        #      = ( (W_E^T @ GZ^T ) @ S ) @ X
        # GW_V = ( (W_E^T @ ( P^T - Y^T ) ) @ S ) @ X


        # COMMENTED OUT AS GO IS CALCULATED IN FFL GRADIENT CALCULATIONS
        # Calculating GO 
        # self.GO = self.GZ @ self.embedding_matrix   # (B, T, H)

        # Calculating GV
        self.GV = self.S.transpose(-2, -1) @ self.GO    # (B, T, H)

        # Calculating GW_V
        self.GW_V = self.GV.transpose(-2, -1) @ X
        self.GW_V = self.GW_V.sum(dim=0)           # Sum over B (H, C)
        return self.GW_V

        ###
        ### OLD IMPLEMENTATION * INCORRECT DERIVATION *
        ###
        # # GO = (P - Y) @ W_lm                          - GO --> (B, T, Vocab size) @ (Vocab size, C) --> (B, T, C)
        # # O = A @ V so linear derivative property says - GV = A.T @ GO   --> (B, T, T) @ (B, T, C) --> (B, T, C)
        # # V = X @ W_V                                  - GW_V = X.T @ GV --> (B, C, T) @ (B, T, C) --> (B, C, C)
        # # So GW_V = X.T ( A.T @ [( P - Y) @ W_lm])     - (B, C, T) @ [ (B, T, T) @ [ (B, T, Vocab Size) @ (Vocab Size, C) ] ]
        # #                                              - (B, C, T) @ [ (B, T, T) @ [ (B, T, C)]]
        # #                                              - (B, C, T) @ [ (B, T, C)]
        # #                                              - (B, C, C)      * Consistent *
        # self.P = torch.clamp(self.P, 1e-9, 1.0)
        # self.GZ = self.P.clone()
        # B_indices = torch.arange(self.B, device=device).view(-1, 1)    # (B, 1)
        # T_indices = torch.arange(self.T, device=device).view(1, -1)    # (1, T)
        # self.GZ[B_indices, T_indices, y] -= 1                # (B, T, Vocab Size)

        # GO = self.GZ @ self.embedding_matrix     # (B, T, C)
        # self.GV = self.A.transpose(-2, -1) @ GO  # (B, T, C)
        # GW_V = X.transpose(-2, -1) @ self.GV     # (B, C, C)

        # # We need sum of gradients across batches
        # GW_V = GW_V.sum(dim=0)              # (C, H)
        # return GW_V.T                       # (H, C) for updating W_V which is also (H, C)

    def gradient_W_qk(self, X):
        # X                                 - (B, T, C)
        # GZ = P - Y                        - (B, T, Vocab size)
        # W_lm or embedding matrix          - (Vocab size, C)
        # GO = GZ @ W_lm                    - (B, T, C) - Only 'C' because C equals H

        ### K and Q                         - (B, T, H)
        # W_k and W_q                       - (H, C)
        # Q = X @ W^T_q                     - (B, T, H)
        # K = X @ W^T_k                     - (B, T, H)
        # GS = GO @ V^T                     - (B, T, T)
        # GA = S * (GS - S dot GS)          - (B, T, T)
        # GQ = 1/root(H) * GA @ K           - (B, T, H)
        # GK = 1/root(H) * GA^T @ Q         - (B, T, H)
        # GW_Q = GQ^T @ X                   - (H, C)
        # GW_K = GK^T @ X                   - (H, C)
        
        # Calculating GS
        self.GS = self.GO @ self.V.transpose(-2, -1)

        # Calculating GA
        S_GS = self.S * self.GS
        rowsum = S_GS.sum(dim=-1, keepdim=True) # (B, T, 1)
        self.GA = self.S * (self.GS - rowsum)   # (B, T, T)

        # Calculating GQ and GK
        # print(f"GQ = 1/root(h) * GA @ X:\t ({self.GA.shape}^T @ {self.K.shape})\n")
        self.GQ = (1 / torch.sqrt(torch.tensor(self.H, dtype=torch.float32))) * (self.GA @ self.K)
        self.GK = (1 / torch.sqrt(torch.tensor(self.H, dtype=torch.float32))) * (self.GA.transpose(-2, -1) @ self.Q)

        # Calculating W_Q and W_K       
        self.GW_Q = self.GQ.transpose(-2, -1) @ X
        self.GW_K = self.GK.transpose(-2, -1) @ X

        # Sum out over B
        self.GW_Q = self.GW_Q.sum(dim=0)           # (H, C)
        self.GW_K = self.GW_K.sum(dim=0)           # (H, C)

        return self.GW_Q, self.GW_K

        ###
        ### OLD IMPLEMENTATION * INCORRECT DERIVATION *
        ###
        # GO = self.GZ @ self.embedding_matrix    # (Vocab size, C)
        # GA = GO @ self.V.transpose(-2, -1)      # (B, T, T)
        # A_GA = self.A * GA                      # (B, T, T) - Element wise multiplication
        # rowsums = A_GA.sum(dim=-1, keepdim=True)# (B, T, 1)
        # GS = self.A * (GA - rowsums)            # (B, T, T)
        # GS = GS / (self.H ** 0.5)            # Scaling to prevent crazy value growth

        # # Derive gradients for Q and K weights
        # self.GQ = GS @ self.K                        # (B, T, H)
        # self.GK = GS.transpose(-2, -1) @ self.Q      # (B, T, H)

        # GW_Q = self.GQ.transpose(-2, -1) @ X         # (B, H, C) or (B, C, C)
        # GW_K = self.GK.transpose(-2, -1) @ X         # (B, H, C) or (B, C, C)
        
        # GW_Q = GW_Q.sum(dim=0)                  # (H, C)
        # GW_K = GW_K.sum(dim=0)                  # (H, C)

        # return GW_Q, GW_K
    
    def gradient_W_e(self, x_ids):
        # O                 - (B, T, C)
        # Z = O @ W^T_e     - (B, T, Vocab size)
        # GZ = P - Y        - (B, T, Vocab Size)
        ## Output gradient ##
        # GW_e = (GZ)^T @ O - (B, Vocab size, C)
        ## Input gradient ##
        # GX = GQ @ W_Q + GK @ W_K + GV @ W_V   - (B, T, C)
        
        # NEED TO CHANGE THIS TO O_HAT AS THAT IS USED FOR CALCULATING LOGITS RATHER THAN O
        # Output gradient
        GZ_flat = self.GZ.view(-1, self.vocab_size) # Makes (B, T, Vocab size) --> (B * T, Vocab size)
        # CHANGED TO USE O_HAT INSTEAD OF O
        O_flat = self.O_hat.view(-1, self.C)            # Makes (B, T, C) --> (B * T, C)

        GW_E_output = GZ_flat.T @ O_flat    # (Vocab size, B*T) @ (B*T, C) --> (Vocab size, C)

        # Input gradient
        self.GX = (self.GQ @ self.W_q) + (self.GK @ self.W_k) + (self.GV @ self.W_v) # (B, T, C)
        GW_E_input = torch.zeros_like(self.embedding_matrix)    # (Vocab size, C)
        GW_E_input.index_add_(0, x_ids.to(device).view(-1), self.GX.view(-1, self.C))

        return GW_E_output + GW_E_input

    # Gradients for weights within the Feed-forward layer
    def gradient_ffl(self):
        # Gradient of O_hat
        #             (B, T, Vocab) @ (Vocab, C)
        self.GO_hat = self.GZ @ self.embedding_matrix   # (B, T, C)

        ### Gradients for second layer
        # W_FF2                    (C, B*T)                                               (B*T, 4C))
        self.GW_ff2 = torch.flatten(self.GO_hat, start_dim=0, end_dim=1).T @ torch.flatten(self.O_ff2, start_dim=0, end_dim=1)  # (C, 4C)

        # b_ff2
        self.Gb_ff2 = torch.flatten(self.GO_hat, start_dim=0, end_dim=1).sum(dim=0)     # (C,)

        # O_ff2       (B, T, C)     (C, 4C)
        self.GO_ff2 = self.GO_hat @ self.W_ff2      # (B, T, 4C)

        ### Gradients for first layer
        # O_ff1 - Derivative of ReLU is a binary mask (0 or 1)
        GReLU = torch.where(self.O_ff1 >= 0, 1.0, 0.0)
        self.GO_ff1 = self.GO_ff2 * GReLU       # B, T, 4C)

        # W_ff1                    (4C, B*T)                                              (B*T, C)
        self.GW_ff1 = torch.flatten(self.GO_ff1, start_dim=0, end_dim=1).T @ torch.flatten(self.O, start_dim=0, end_dim=1)  # (4C, C)

        # b_ff1
        self.Gb_ff1 = torch.flatten(self.GO_ff1, start_dim=0, end_dim=1).sum(dim=0) # (4C,)

        # O       (B, T, 4C)    (4C, C)
        self.GO = self.GO_ff1 @ self.W_ff1  # (B, T, C)

        # We do not need to sum them over B because that was already done. We will average them out in back_pass()
        return self.GW_ff1, self.Gb_ff1, self.GW_ff2, self.Gb_ff2

    # Adam optimizer moments calculations and weight update
    def adam_step(self, W, grad, m, v):
        m = self.beta1 * m + (1.0 - self.beta1) * grad          # First moment (beta1 = 0.9)
        v = self.beta2 * v + (1.0 - self.beta2) * (grad ** 2)   # Second moment (beta2 = .999)

        # Bias correction
        m_hat = m / (1.0 - (self.beta1 ** self.t))
        v_hat = v / (1.0 - (self.beta2 ** self.t))

        # Weight update
        W = W - self.lr * m_hat / (torch.sqrt(v_hat) + self.epsilon)

        return W, m, v

    def back_pass(self, x_ids, X, y):
        # Increment step counter for bias correction
        self.t += 1

        # Getting GZ before calculating other gradients
        self.P = torch.clamp(self.P, 1e-9, 1.0) # Enforce a min/max of 1e-9/1.0
        self.GZ = self.P.clone()    # Copy of P (will be used to subtract one hot vector)
        B_indices = torch.arange(self.B, device=device).view(-1, 1)    # (B, 1)
        T_indices = torch.arange(self.T, device=device).view(1, -1)    # (1, T)
        self.GZ[B_indices, T_indices, y] -= 1                # (B, T, Vocab Size)

        # Get gradients
        GW_ff1, Gb_ff1, GW_ff2, Gb_ff2 = self.gradient_ffl()
        GW_V = self.gradient_W_v(X)
        GW_Q, GW_K = self.gradient_W_qk(X)
        GW_E = self.gradient_W_e(x_ids)

        # Update weights
        grad_v = GW_V / self.B
        # print(f"W_q: ({self.W_q.shape})\nlr: ({self.lr})\nGW_Q: ({GW_Q.shape})\nBT: ({BT})")
        grad_q = GW_Q / self.B
        grad_k = GW_K / self.B
        grad_e = GW_E / self.B

        BT = self.B * self.T
        GW_ff1 = GW_ff1 / BT
        Gb_ff1 = Gb_ff1 / BT
        GW_ff2 = GW_ff2 / BT
        Gb_ff2 = Gb_ff2 / BT

        # Setting up posiitional embbeding gradient + summing over Batch
        grad_pos = self.GX.sum(dim = 0)   # Sum over B (T, C)
        grad_pos = grad_pos / self.B

        # Apply adam step to all weight matrices
        self.W_v, self.m_Wv, self.v_Wv = self.adam_step(self.W_v, grad_v, self.m_Wv, self.v_Wv)
        self.W_q, self.m_Wq, self.v_Wq = self.adam_step(self.W_q, grad_q, self.m_Wq, self.v_Wq)
        self.W_k, self.m_Wk, self.v_Wk = self.adam_step(self.W_k, grad_k, self.m_Wk, self.v_Wk)
        self.embedding_matrix, self.m_Emb, self.v_Emb = self.adam_step(self.embedding_matrix, grad_e, self.m_Emb, self.v_Emb)
        self.W_pos, self.m_Wpos, self.v_Wpos = self.adam_step(self.W_pos, grad_pos, self.m_Wpos, self.v_Wpos)

        self.W_ff1, self.m_Wff1, self.v_Wff1 = self.adam_step(self.W_ff1, GW_ff1, self.m_Wff1, self.v_Wff1)
        self.b_ff1, self.m_bff1, self.v_bff1 = self.adam_step(self.b_ff1, Gb_ff1, self.m_bff1, self.v_bff1)
        self.W_ff2, self.m_Wff2, self.v_Wff2 = self.adam_step(self.W_ff2, GW_ff2, self.m_Wff2, self.v_Wff2)
        self.b_ff2, self.m_bff2, self.v_bff2 = self.adam_step(self.b_ff2, Gb_ff2, self.m_bff2, self.v_bff2)

    ##### TRAINING LOOP #####
    def train(self, data, max_iter=10000):  # Default Iterations = 10k
        for i in tqdm(range(max_iter), desc="LLM Training", total=max_iter):
            # Get random batches
            x_ids, X, y = self.build_train_batch(data)

            self.forward_pass(X)
            self.back_pass(x_ids, X, y)

            if i % 100 == 0:
                # Manual Cross-Entropy Loss: -log(probability of the correct token)
                B_idx = torch.arange(self.B).view(-1, 1)
                T_idx = torch.arange(self.T)
                correct_probs = self.P[B_idx, T_idx, y]
                loss = -torch.log(correct_probs + 1e-9).mean() # 1e-9 prevents log(0)
                print(f"Iter {i}: Loss {loss.item():.4f}")
                #print(f"\n{self.W_q, self.W_k, self.W_v, self.embedding_matrix}\n")
                
        return self.W_q, self.W_k, self.W_v, self.embedding_matrix





In [43]:
processor = Processor()

# Read and ingest data
with open("tiny-shakespeare.txt", 'r') as f:
  data = f.read()

# data = "Hello my name is Kylan. What is your father doing out here in the cold?"
processor.ingest(data)
tokenized_data_list = processor.encode(data)

# Convert into tensor
tokenized_data = torch.tensor(tokenized_data_list, dtype=torch.long)

# Get vocab size
vocab_size = processor.vocab_size

In [44]:
model = Model(vocab_size, lr=1e-3)
model.train(tokenized_data)
model.save_weights()


LLM Training:   0%|          | 3/10000 [00:00<07:04, 23.53it/s]

Iter 0: Loss 4.2360


LLM Training:   1%|          | 105/10000 [00:03<06:07, 26.89it/s]

Iter 100: Loss 3.0782


LLM Training:   2%|▏         | 204/10000 [00:07<06:07, 26.67it/s]

...

Iter 4900: Loss 4.1744


LLM Training:  50%|█████     | 5003/10000 [03:20<03:07, 26.59it/s]

Iter 5000: Loss 4.1744


LLM Training:  51%|█████     | 5055/10000 [03:22<03:18, 24.94it/s]


KeyboardInterrupt: 

In [ ]:
loaded_model = Model(vocab_size, lr=1e-3)

# Load the saved parameters
loaded_model.load_weights()

# Continue training
loaded_model.train(tokenized_data)

In [41]:
new_model = Model(vocab_size, lr=1e-3)

# Load the saved parameters
new_model.load_weights()

text = """
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather
"""
generated_text = new_model.generate(text, processor, max_new_tokens=4000)
print(f"Generated output:\n{generated_text}")

Weights successfully loaded from model_weights.pt
Generated output:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather
lost the reve;
Thoul, and have the your be sthreals,
And hobgeal
f Masce.

CLARIANA:
Opsciout Lacion, griout ound anll agrie trive.

PRICHARD IANT:
Whap but main! woul, my muke; I old, forth
Lord you!Coupsor ithen quee your But Sichy me't kake my you evence, lifl; hegent is whim of wore Englaled;
God be sweer
A Livep, welly gabally if
My I know your rom of, you ar athe do?
This spceas; upeack, sid am your wrenfem.

GRAMIANDIV:
Your kwelligh'd
The have theer, sibles ontriedr'd of our maffusn,
And to-ther:
He Some is det!

YORK:
Nather mours Oxcentor bead! dee, Senamass, nook somind, bead ayaved with hee,
satignencre so ould tt by nordsuld of your outeme,
Pauted heast by all in fartak thim us virst,
Who me lerce: I'll wicknemeved whals mew!

HORMEO:
The can the willy fagartion, mand!


DUCHSOTRESS